In [1]:
import numpy as np
from scipy.optimize import root, linprog

# Метод множителей Лагранжа

### **Идея метода множителей Лагранжа**

Метод множителей Лагранжа — это способ решения задач **условной оптимизации** вида:
$\min f(x) \quad \text{при условии} \quad h(x) = 0.$

#### **1. Формулировка**
Вводится **функция Лагранжа**:
$\mathcal{L}(x, \lambda) = f(x) + \lambda h(x),$
где $ \lambda $ — множитель Лагранжа.

#### **2. Необходимые условия экстремума**
Точка $ x^* $ может быть экстремумом, если:
$
\nabla \mathcal{L} = 0 \quad \Rightarrow \quad 
\begin{cases}
\nabla f(x) + \lambda \nabla h(x) = 0, \\
h(x) = 0.
\end{cases}
$

Эту систему уравнений и решает данный код.

#### **3. Геометрическая интерпретация**
В точке экстремума градиенты $ \nabla f $ и $ \nabla h $ **коллинеарны** (параллельны), поэтому:
$
\nabla f = -\lambda \nabla h.
$

In [2]:
def lagrange_multiplier(f, h, grad_f, grad_h, x0, lambda0, tol=1e-6):
    def equations(vars):
        x = vars[:-1]
        lam = vars[-1]
        return np.concatenate([grad_f(x) + lam * grad_h(x), [h(x)]])
    
    vars0 = np.concatenate([x0, [lambda0]])
    result = root(equations, vars0, tol=tol)
    if not result.success:
        raise ValueError("Решение не найдено")
    x_opt = result.x[:-1]
    lambda_opt = result.x[-1]
    return x_opt, lambda_opt

# Метод Зойтендейка

## **Идея метода Зойтендейка**
Метод Зойтендейка — это **метод возможных направлений** для задач вида:
$
\min f(x) \quad \text{при} \quad g_i(x) \leq 0, \quad i = 1, \dots, m.
$

#### 1. **Направление спуска**:
   - В каждой точке ищем направление `d`, которое:
     - Уменьшает целевую функцию ($ \nabla f^T d < 0 $).
     - Не выходит за активные ограничения ($ \nabla g_i^T d \leq 0 $).
   - Если нет активных ограничений, движемся вдоль антиградиента (`d = -∇f`).

#### 2. **Линейное программирование**:
   - Направление `d` находится решением задачи ЛП:
     $
     \min \nabla f^T d \quad \text{при} \quad \nabla g_i^T d \leq 0, \quad \|d\|_\infty \leq 1.
     $

#### 3. **Поиск шага**:
   - Используется **поиск шага с откатом** (поиск шага, если условия не выполнены поиск шага меньше), чтобы оставаться в допустимой области.

#### 4. **Условия остановки**:
   - Направление `d` близко к нулю (нет возможного спуска).
   - Градиент `∇f` мал (достигнут локальный минимум).

Метод Зойтендейка более универсален так как подходит для задач с **неравенствами**, тогда как метод Лагранжа только для **равенств**.

In [3]:
def zoitenjik(f, grad_f, constraints, grad_constraints, x0, alpha=0.5, beta=0.5, max_iter=100, tol=1e-6):
    x = np.array(x0, dtype=float)
    history = [x.copy()]
    
    for iter in range(max_iter):
        active = [i for i, constr in enumerate(constraints) if constr(x) >= -tol]
        c = grad_f(x)

        if not active:
            d = -c
        else:
            A = np.array([grad_constraints[i](x) for i in active])
            b = np.zeros(len(A))
            res = linprog(c, A_ub=A, b_ub=b, bounds=[(-1, 1)]*len(x))
            d = res.x if res.success else np.zeros_like(x)

        if np.linalg.norm(d) < tol or np.linalg.norm(c) < tol:
            break

        t = 1.0
        for _ in range(100):
            x_new = x + t * d
            if all(constr(x_new) <= tol for constr in constraints) and f(x_new) < f(x):
                break
            t *= beta
        else:
            break
        
        x = x_new
        history.append(x.copy())

        if iter > 0 and np.linalg.norm(history[-1] - history[-2]) < 1e-8:
            break
    
    return x, np.array(history)

# Метод проекции градиента Розена

## **Идея метода проекции градиента**
Метод решает задачу:
$
\min f(x) \quad \text{при условии} \quad x \in C,
$
где $ C $ — замкнутое выпуклое множество.

#### 1. **Градиентный шаг**:
   - Сначала делаем шаг градиентного спуска:
     $
     x_{k+1}^{\text{unconstr}} = x_k - \alpha \nabla f(x_k).
     $
#### 2. **Проекция**:
   - Затем проектируем на допустимое множество:
     $
     x_{k+1} = P_C(x_{k+1}^{\text{unconstr}}),
     $
     где $ P_C $ — оператор проекции:
     $
     P_C(z) = \arg\min_{x \in C} \|x - z\|^2.
     $

#### 3. **Условие остановки**:
   - Алгоритм останавливается, когда `norm(x_new - x) < tol` (малое изменение).

По сути метод Розена - усовершенствованный метод градиентного спуска для задач с ограничениями

In [4]:
def projected_gradient(f, grad_f, projection, x0, alpha=0.1, max_iter=1000, tol=1e-6):
    x = np.array(x0, dtype=float)
    history = [x.copy()]
    
    for _ in range(max_iter):
        grad = grad_f(x)
        x_new = projection(x - alpha * grad)
        if np.linalg.norm(x_new - x) < tol:
            break
        x = x_new
        history.append(x.copy())
    
    return x, np.array(history)

# Проверка алгоритмов

In [5]:
f = lambda x: x[0]**2 + x[1]**2
grad_f = lambda x: np.array([2*x[0], 2*x[1]])

def test_lagrange():
    h = lambda x: x[0] - 1
    grad_h = lambda x: np.array([1, 0])
    
    x_opt, lambda_opt = lagrange_multiplier(f, h, grad_f, grad_h, x0=[0.0, 0.0], lambda0=0)
    print("Метод Лагранжа:", list(map(lambda x: round(x, 4), x_opt)))
    print(f"{round(f(x_opt), 4)}")

def test_projected_gradient():
    projection = lambda x: np.array([max(x[0], 1), x[1]])
    
    x_opt, _ = projected_gradient(f, grad_f, projection, x0=[2.0, 2.0], alpha=0.1)
    print("Метод проекции градиента:", list(map(lambda x: round(x, 4), x_opt)))
    print(f"{round(f(x_opt), 4)}")

def test_zoitenjik():
    constraints = [
        lambda x: -x[0] + 1
    ]
    grad_constraints = [
        lambda x: np.array([-1, 0])
    ]
    
    x_opt, _ = zoitenjik(f, grad_f, constraints, grad_constraints, x0=[2.0, 2.0])
    print("Метод Зойтендейка:", list(map(lambda x: round(x, 4), x_opt)))
    print(f"{round(f(x_opt), 4)}")

print("Тестирование на задаче f(x) = x1² + x2² при ограничении x1 >= 1 (x1 = 1 для Лагранжа):")
test_lagrange()
test_projected_gradient()
test_zoitenjik()

Тестирование на задаче f(x) = x1² + x2² при ограничении x1 >= 1 (x1 = 1 для Лагранжа):
Метод Лагранжа: [1.0, 0.0]
1.0
Метод проекции градиента: [1.0, 0.0]
1.0
Метод Зойтендейка: [1.0, 0.0]
1.0
